# Arabic Text Simplification - Unified Inference Notebook
## Group 15 - ALLaM / Fanar

**Dataset:** 40 samples (P001-P040)\n**Prompt:** Unified few-shot (2 examples) for fair comparison\n**Models:** ALLaM (7B), Fanar (9B)

## Cell 1: Install Dependencies

In [ ]:
!pip install transformers accelerate torch bitsandbytes -q

## Cell 2: Imports & Utilities

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch
import json
import re
import gc
import os

print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

GPU: Tesla T4
GPU Memory: 15.6 GB


## Cell 3: Load Dataset

In [ ]:
# Upload dataset.json to Colab first, then run this cell
with open('dataset.json', 'r', encoding='utf-8') as f:
    dataset = json.load(f)

print(f"Loaded {len(dataset)} samples")
for item in dataset[:3]:
    print(f"{item['ID']}: {item['arabic_type']} | {item['domain']}")

Loaded 40 samples
P001: MSA | health
P002: MSA | education
P003: MSA | environment


## Cell 4: Define Few-Shot Prompt & Utilities

**Same prompt for ALL models** - 2 examples (1 MSA + 1 Dialect)

In [ ]:
# Few-shot examples (taken from dataset)
FEW_SHOT_EXAMPLES = """
مثال 1:
النص الأصلي: تُعدّ السمنة من أكثر المشكلات الصحية انتشاراً في العالم العربي خلال العقود الأخيرة، وترتبط ارتباطاً وثيقاً بالإصابة بأمراض القلب والسكري وارتفاع ضغط الدم.
النص المبسط: السمنة مشكلة صحية شائعة في العالم العربي. وهي تسبب أمراضاً كالسكري وأمراض القلب. أسبابها الرئيسية هي الأكل غير الصحي وقلة الحركة. والحل هو الغذاء الصحي وممارسة الرياضة.

مثال 2:
النص الأصلي: والله يا جماعة الموضوع ده كبير وبيشغل بالي أوي. إحنا في مصر بقينا ناس التليفون مش بيتنحّى من إيدينا، الواحد صاحي ينام، التليفون في إيده.
النص المبسط: الناس دلوقتي مش قادرين يبعدوا عن التليفون، الكبار والصغار. وده بيأثر على علاقاتنا ببعض، لأن الواحد بيبقى جنب ناس بس عقله في موبايله.
"""

def build_prompt(original_text):
    """Build unified few-shot prompt for any model."""
    return f"""قم بتبسيط النصوص العربية التالية. اجعلها أسهل في القراءة والفهم مع الحفاظ على المعنى الأصلي.

{FEW_SHOT_EXAMPLES}

الآن، بسّط النص التالي:
النص الأصلي: {original_text}
النص المبسط:"""

def clean_output(text):
    """Clean residual artifacts."""
    text = text.replace("▁", " ")
    text = text.replace("<0x0A>", "\n")
    text = re.sub(r'\[/\s*IN\s*ST\s*\]', '', text)
    text = re.sub(r'\[\s*IN\s*ST\s*\]', '', text)
    text = re.sub(r'<<\s*SYS\s*>>', '', text)
    text = text.replace("<|eot_id|>", "")
    text = text.replace("<|assistant|>", "")
    text = text.replace("<|user|>", "")
    text = text.replace("<|system|>", "")
    text = text.replace("<|endoftext|>", "")
    text = re.sub(r' +', ' ', text)
    text = re.sub(r'\n+', '\n', text)
    return text.strip()

def simplify_one(original_text, model, tokenizer, device, max_new_tokens=200):
    """Run inference on a single sample."""
    prompt = build_prompt(original_text)

    inputs = tokenizer(prompt, return_tensors="pt", return_token_type_ids=False)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    input_length = inputs['input_ids'].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.3,
            top_p=0.95,
            top_k=50,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated_tokens = outputs[0][input_length:]
    generated_text = tokenizer.batch_decode(
        [generated_tokens],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True
    )[0]

    return clean_output(generated_text)

def run_model(model_name, model, tokenizer, device):
    """Run inference on all 40 samples."""
    results = {}
    for i, item in enumerate(dataset):
        sid = item['ID']
        out = simplify_one(item['original'], model, tokenizer, device)
        results[sid] = out
        print(f"{sid}: {out[:60]}...")

        if (i + 1) % 10 == 0:
            gc.collect()
            torch.cuda.empty_cache()

    # Save
    outfile = f"predictions_{model_name}.json"
    with open(outfile, 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    print(f"\n✅ Saved {len(results)} predictions to {outfile}")
    return results

---
## MODEL 1: ALLaM (7B)
**Run this cell, then wait for download + inference**

In [ ]:
# ============================================
# MODEL 1: ALLaM (OFFICIAL CORRECT WAY)
# ============================================
MODEL_NAME = "ALLaM"
HF_NAME = "ALLaM-AI/ALLaM-7B-Instruct-preview"

print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(HF_NAME, trust_remote_code=True)

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

model = AutoModelForCausalLM.from_pretrained(
    HF_NAME,
    device_map="auto",
    quantization_config=quantization_config,
    trust_remote_code=True,
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

device = next(model.parameters()).device
print(f"Model loaded on {device}")

# Run inference - OFFICIAL PATTERN
results = {}
for i, item in enumerate(dataset):
    sid = item['ID']

    # Build messages
    messages = [
        {"role": "user", "content": f"بسّط هذا النص العربي:\n\n{item['original']}\n\nالنص المبسط:"}
    ]

    # Apply chat template to get formatted string
    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # Tokenize the formatted string
    inputs = tokenizer(formatted_prompt, return_tensors="pt", return_token_type_ids=False)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # Generate
    with torch.no_grad():
        response_ids = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=True,
            temperature=0.6,
            top_k=50,
            top_p=0.95,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    # Decode FULL output (not sliced!)
    decoded = tokenizer.batch_decode(response_ids, skip_special_tokens=True)[0]

    # Strip the input prompt from the beginning
    if decoded.startswith(formatted_prompt):
        output = decoded[len(formatted_prompt):].strip()
    else:
        output = decoded.strip()

    results[sid] = output
    print(f"{sid}: {output[:60]}...")

    if (i + 1) % 10 == 0:
        gc.collect()
        torch.cuda.empty_cache()

# Save
with open("predictions_ALLaM.json", 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"\n✅ Saved {len(results)} predictions")

# Clear memory
del model, tokenizer
gc.collect()
torch.cuda.empty_cache()

Loading ALLaM...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Model loaded on cuda:0
P001: [ IN ST ] بس ّط هذا ال نص ال عر بي : تُ عد ّ ال سم نة من أكث...
P002: [ IN ST ] بس ّط هذا ال نص ال عر بي : يمر ّ الن ظام ال تع لي ...
P003: [ IN ST ] بس ّط هذا ال نص ال عر بي : تو اجه من طقة ال شرق ال...
P004: [ IN ST ] بس ّط هذا ال نص ال عر بي : أ حد ثت الث ورة ال رق م...
P005: [ IN ST ] بس ّط هذا ال نص ال عر بي : تس عى دول ال خل يج ال ع...
P006: [ IN ST ] بس ّط هذا ال نص ال عر بي : ي ُم ث ّل م فه وم الح و...
P007: [ IN ST ] بس ّط هذا ال نص ال عر بي : يُ سب ّب الت وتر ال مز ...
P008: [ IN ST ] بس ّط هذا ال نص ال عر بي : با تت قض ية الم سا واة ...
P009: [ IN ST ] بس ّط هذا ال نص ال عر بي : يع كف عل ماء الأ حي اء ...
P010: [ IN ST ] بس ّط هذا ال نص ال عر بي : يش هد قط اع الإ عل ام ا...
P011: [ IN ST ] بس ّط هذا ال نص ال عر بي : اض طل عت الح ضا رة الإ ...
P012: [ IN ST ] بس ّط هذا ال نص ال عر بي : تو اجه الم دن ال عر بية...
P013: [ IN ST ] بس ّط هذا ال نص ال عر بي : يُ عا ني قط اع ال زر اع...
P014: [ IN ST ] بس ّط هذا ال نص ال عر بي : ي ُم ث ّل ال شب اب ال ع.

In [ ]:
# ============================================
# MODEL 2: Fanar
# ============================================
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = "Fanar"
HF_NAME = "QCRI/Fanar-1-9B-Instruct"

print(f"\nLoading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(HF_NAME)

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

model = AutoModelForCausalLM.from_pretrained(
    HF_NAME,
    device_map="auto",
    quantization_config=quantization_config,
    trust_remote_code=True,
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
if model.config.pad_token_id is None:
    model.config.pad_token_id = model.config.eos_token_id

print(f"Model loaded on {next(model.parameters()).device}")
print(f"GPU memory used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

fanar_results = run_model(MODEL_NAME, model, tokenizer, next(model.parameters()).device)

print("\n✅ Fanar complete! Saved to predictions_Fanar.json")